# ✦ DianaLuxe — Serveur vidéo IA gratuit (Colab / Kaggle)

Ce notebook lance le **serveur DianaLuxe** sur un GPU gratuit et le rend accessible sur Internet.

**Moteurs (dans l'ordre) :**
1. **Agnes AI** si vous mettez une ou plusieurs clés gratuites (platform.agnes-ai.com). Gestion automatique de la saturation.
2. **LTX-Video** (open-source, Lightricks) sur le GPU du notebook : aucune limite, aucun compte.

**Mode d'emploi :**
- Colab : *Exécution → Modifier le type d'exécution → GPU T4*. Kaggle : *Settings → Accelerator → GPU T4 x2* et *Internet : On*.
- Exécutez les cellules dans l'ordre, puis copiez l'**adresse** et le **jeton** affichés dans DianaLuxe (⚙️ Serveur IA).


In [ ]:
#@title 1. Installation (2-3 min)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Pas de GPU : seul le moteur Agnes fonctionnera"
!pip install -q -U "diffusers>=0.33" "transformers>=4.44" accelerate sentencepiece ftfy imageio imageio-ffmpeg fastapi uvicorn httpx
import os, subprocess
if not os.path.exists('NEWAI'):
    r = subprocess.run(['git','clone','-q','--depth','1','-b','claude/busy-mccarthy-m4umsf','https://github.com/dianahilali3-prog/NEWAI'])
    if r.returncode != 0:
        print("❌ Clonage impossible (dépôt privé ?). Envoyez le dossier 'server' dans les fichiers du notebook sous NEWAI/server.")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print("✅ Installation terminée")

In [ ]:
#@title 2. Configuration
# Clés Agnes (optionnel) : séparées par des virgules. Laissez vide pour n'utiliser que le GPU.
# Sur Colab vous pouvez aussi créer un secret nommé AGNES_API_KEYS (icône 🔑).
AGNES_API_KEYS = ""  #@param {type:"string"}
LOCAL_MODEL = "ltx"  #@param ["ltx", "wan"]
try:
    from google.colab import userdata
    AGNES_API_KEYS = AGNES_API_KEYS or (userdata.get('AGNES_API_KEYS') or '')
except Exception:
    pass
import secrets, torch
backends = []
if AGNES_API_KEYS.strip(): backends.append('agnes')
if torch.cuda.is_available(): backends.append('local')
TOKEN = secrets.token_urlsafe(16)
os.environ.update(AGNES_API_KEYS=AGNES_API_KEYS, LOCAL_MODEL=LOCAL_MODEL, DIANALUXE_TOKEN=TOKEN,
                  DIANALUXE_BACKENDS=','.join(backends))
print("Moteurs :", backends or "❌ aucun — ajoutez une clé Agnes ou activez le GPU")

In [ ]:
#@title 3. Démarrer le serveur + lien public
import subprocess, time, re
server = subprocess.Popen(['python','-m','uvicorn','app:app','--host','0.0.0.0','--port','8000'],
                          cwd='NEWAI/server', stdout=open('server.log','w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                          stdout=open('tunnel.log','w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: url = m.group(0); break
    time.sleep(1)
time.sleep(3)
print("\n" + "═"*60)
print("✦ Adresse du serveur :", url)
print("✦ Jeton             :", TOKEN)
print("═"*60)
print("Collez-les dans DianaLuxe → ⚙️ Serveur IA. Gardez ce notebook ouvert.")
print("Le modèle se télécharge en arrière-plan au 1er lancement (~5-10 min, voir cellule 4).")

In [ ]:
#@title 4. Journal du serveur (relancez pour actualiser)
!tail -n 30 server.log